# 🚀 Google Colab Pro Scheduled Runner (Dual-Stream Ingestion & Training)
### Hỗ Trợ 100% Option C: Chạy Trực Tiếp Từ Google Drive (Không Cần Xác Thực GitHub)

Notebook này được tối ưu hóa cho **Google Colab Pro Scheduled Runs (100% Headless)**:
- 📂 **Ưu Tiên Nạp Mã Nguồn Từ Google Drive (Option C)**: Tự động phát hiện mã nguồn tại `/content/drive/MyDrive/Thread_Data/DeepFake-Detection-Agent/` hoặc tệp nén `DeepFake-Detection-Agent.zip`, hoàn toàn không phụ thuộc vào GitHub Private Token.
- 🎛️ **Giao Diện Form Điều Khiển Tương Tác (Option B #@param)**: Cho phép nghiên cứu viên linh hoạt tùy chỉnh tham số, bật/tắt Model 1 hoặc Model 2 khi chạy thủ công.
- 🔄 **Phân Tách 2 Mô Hình Độc Lập**:
  1. **Mô Hình 1 (General Vision & Signal)**: Giữ tập dataset tổng quát (`NTIRE`, `CIFAKE`, `DF40`, `DFFD`) + Tăng cường mẫu biên khó ($H(P) \ge 0.70$).
  2. **Mô Hình 2 (Human Biometrics & Physiology)**: Bổ sung 4 dataset human chuyên sâu (`MultiFakeVerse`, `HiDF`, `140k Faces`, `DeepDetect-2025/2026`) + Dữ liệu 5 phân lớp người Việt.
- 📊 **Báo Cáo Toàn Diện**: Tự động lưu `training_report_dual_stream.md` về Google Drive.

--- 
## Bước 1: Kết Nối Google Drive (Auto-Mount Without Pop-up)

In [ ]:
import os, sys, time, shutil, zipfile
from pathlib import Path
from google.colab import drive

# Auto-mounts Google Drive at /content/drive/MyDrive without pop-up blocking
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive đã được gắn kết thành công tại /content/drive/MyDrive')

--- 
## Bước 2: Nạp Mã Nguồn Theo Option C (Google Drive) Hoặc GitHub Fallback

In [ ]:
LOCAL_REPO_DIR = Path("/content/DeepFake-Detection-Agent")
GDRIVE_THREAD_DATA = Path("/content/drive/MyDrive/Thread_Data")

# Danh mục các vị trí mã nguồn khả dĩ trên Google Drive (Option C)
GDRIVE_CANDIDATES = [
    GDRIVE_THREAD_DATA / "DeepFake-Detection-Agent",
    GDRIVE_THREAD_DATA / "source_code" / "DeepFake-Detection-Agent",
    Path("/content/drive/MyDrive/DeepFake-Detection-Agent"),
    GDRIVE_THREAD_DATA / "DeepFake-Detection-Agent.zip",
    GDRIVE_THREAD_DATA / "source_code.zip"
]

source_loaded = False

# 1. KIỂM TRA MÃ NGUỒN TRÊN GOOGLE DRIVE (OPTION C ƯU TIÊN)
print("[Bước 2] Đang kiểm tra mã nguồn trên Google Drive (Option C)...\n")
for cand in GDRIVE_CANDIDATES:
    if cand.exists():
        print(f"🎯 [Option C MATCH] Tìm thấy mã nguồn tại Google Drive: {cand}")
        if cand.is_dir() and (cand / "colab_sync_to_gdrive.py").exists():
            if LOCAL_REPO_DIR.exists():
                shutil.rmtree(str(LOCAL_REPO_DIR))
            print(f"  📥 Đang sao chép mã nguồn từ Drive vào máy ảo {LOCAL_REPO_DIR}...")
            shutil.copytree(str(cand), str(LOCAL_REPO_DIR), dirs_exist_ok=True, ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pt', 'venv', '.venv'))
            source_loaded = True
            break
        elif cand.is_file() and cand.suffix.lower() == '.zip':
            print(f"  🗜️ Đang giải nén {cand.name} vào {LOCAL_REPO_DIR}...")
            LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(str(cand), 'r') as zip_ref:
                zip_ref.extractall(str(LOCAL_REPO_DIR))
            # Xử lý trường hợp file zip bọc thêm 1 thư mục cha
            sub_dir = LOCAL_REPO_DIR / "DeepFake-Detection-Agent"
            if sub_dir.exists() and (sub_dir / "colab_sync_to_gdrive.py").exists():
                for item in sub_dir.iterdir():
                    dest_item = LOCAL_REPO_DIR / item.name
                    if not dest_item.exists():
                        shutil.move(str(item), str(dest_item))
            source_loaded = True
            break

# 2. NẾU CHƯA CÓ TRÊN DRIVE: THỬ TẢI QUA GITHUB (KÈM TOKEN NẾU ĐÃ CẤU HÌNH)
if not source_loaded or not (LOCAL_REPO_DIR / "colab_sync_to_gdrive.py").exists():
    print("\n[Bước 2 Fallback] Chưa có mã nguồn trên Google Drive, thử tải từ GitHub...")
    gh_token = ""
    try:
        from google.colab import userdata
        gh_token = userdata.get('GH_TOKEN') or userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    
    if gh_token:
        clone_url = f"https://{gh_token}@github.com/ThangCao106/DeepFake-Detection-Agent.git"
        print("  🔑 Sử dụng GitHub Token từ Colab Secrets để clone...")
    else:
        clone_url = "https://github.com/ThangCao106/DeepFake-Detection-Agent.git"
        
    ret = os.system(f"git clone {clone_url} {LOCAL_REPO_DIR} 2>&1")
    if (LOCAL_REPO_DIR / "colab_sync_to_gdrive.py").exists():
        source_loaded = True
        print("  ✅ Clone thành công từ GitHub!")
        # Tự động sao lưu sang Google Drive để lần sau chạy theo Option C hoàn toàn độc lập
        try:
            drive_backup = GDRIVE_THREAD_DATA / "DeepFake-Detection-Agent"
            if not drive_backup.exists():
                print(f"  💾 Tự động sao lưu mã nguồn sang Google Drive: {drive_backup}...")
                shutil.copytree(str(LOCAL_REPO_DIR), str(drive_backup), dirs_exist_ok=True, ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pt', 'venv', '.venv'))
                print("  ✅ Đã lưu bản sao vào Google Drive thành công!")
        except Exception as ex:
            print(f"  (Ghi chú sao lưu: {ex})")

# 3. KIỂM TRA BẮT BUỘC TRƯỚC KHI THỰC THI
if not (LOCAL_REPO_DIR / "colab_sync_to_gdrive.py").exists():
    error_banner = "\n" + "="*75 + "\n"
    error_banner += "❌ LỖI: KHÔNG TÌM THẤY MÃ NGUỒN ĐỂ THỰC THI!\n"
    error_banner += "Nguyên nhân: Kho GitHub ở chế độ Private và trên Google Drive chưa có mã nguồn.\n\n"
    error_banner += "👉 CÁCH XỬ LÝ THEO OPTION C (KHÔNG CẦN GITHUB - CHỈ MẤT 1 PHÚT):\n"
    error_banner += "  1. Trên máy tính của bạn, tệp 'DeepFake-Detection-Agent.zip' đã được đóng gói sẵn!\n"
    error_banner += "  2. Tải tệp 'DeepFake-Detection-Agent.zip' (hoặc thư mục DeepFake-Detection-Agent)\n"
    error_banner += f"     vào thư mục: MyDrive/Thread_Data/ trên Google Drive của bạn.\n"
    error_banner += "  3. Bấm chạy lại ô này, Colab sẽ tự động nhận diện và giải nén 100%!\n"
    error_banner += "="*75
    raise FileNotFoundError(error_banner)

# Đổi thư mục làm việc và cài đặt dependencies
%cd /content/DeepFake-Detection-Agent
print(f"\n📂 Thư mục làm việc hiện tại: {os.getcwd()}")
!pip install -q -r requirements.txt transformers timm
print('✅ Môi trường và kho mã nguồn đã sẵn sàng 100%!')

--- 
## Bước 3: Form Cấu Hình Tương Tác (Option B Form UI) & Lựa Chọn Tham Số

In [ ]:
#@title 🎛️ Bảng Điều Khiển Huấn Luyện & Thu Thập Dữ Liệu { run: "auto" }

#@markdown ### 🔄 Chế Độ Thực Thi (Execution Mode)
RUN_MODE = "SCHEDULED_BATCH_RUN" #@param ["SCHEDULED_BATCH_RUN", "MANUAL_TEST_RUN", "CONTINUOUS_DAEMON_4H"]
LOOP_INTERVAL = "4h" #@param ["1h", "2h", "4h", "8h", "12h", "24h"]

#@markdown ### 🎯 Lựa Chọn Mô Hình Huấn Luyện (Model Selection)
TRAIN_BOTH_MODELS = True #@param {type:"boolean"}
TRAIN_MODEL_1_GENERAL = True #@param {type:"boolean"}
TRAIN_MODEL_2_HUMAN = True #@param {type:"boolean"}

#@markdown ### 📊 Quota Dữ Liệu In-The-Wild Meta Threads (Mỗi Chu Kỳ)
CRAWL_IMAGES = 10000 #@param {type:"slider", min:100, max:20000, step:100}
CRAWL_VIDEOS = 500 #@param {type:"slider", min:50, max:2000, step:50}
CRAWL_AUDIOS = 500 #@param {type:"slider", min:50, max:2000, step:50}
INSPECTION_SAMPLES = 25 #@param {type:"integer"}

#@markdown ### 🏋️ Siêu Tham Số Huấn Luyện (Hyperparameters)
EPOCHS = 3 #@param {type:"integer"}
BATCH_SIZE = 32 #@param {type:"integer"}
LEARNING_RATE = 0.0001 #@param {type:"number"}
ANTI_IDLE_HEARTBEAT = True #@param {type:"boolean"}

print(f"[Cấu hình] Chế độ: {RUN_MODE} | Crawl: {CRAWL_IMAGES} ảnh, {CRAWL_VIDEOS} video, {CRAWL_AUDIOS} audio")
print(f"[Cấu hình] Huấn luyện: Model 1={TRAIN_MODEL_1_GENERAL}, Model 2={TRAIN_MODEL_2_HUMAN} (Both={TRAIN_BOTH_MODELS})")
print(f"[Cấu hình] Tham số: Epochs={EPOCHS}, BatchSize={BATCH_SIZE}, LR={LEARNING_RATE}")

--- 
## Bước 4: Thực Thi Đồng Bộ Dữ Liệu & Huấn Luyện 2 Mô Hình

In [ ]:
# Xây dựng câu lệnh thực thi dựa trên Form phía trên
cmd_args = [
    f"--images {CRAWL_IMAGES}",
    f"--videos {CRAWL_VIDEOS}",
    f"--audios {CRAWL_AUDIOS}",
    f"--samples {INSPECTION_SAMPLES}",
    f"--epochs {EPOCHS}",
    f"--batch_size {BATCH_SIZE}",
    f"--lr {LEARNING_RATE}"
]

if TRAIN_BOTH_MODELS:
    cmd_args.append("--train-both-models")
else:
    if TRAIN_MODEL_1_GENERAL:
        cmd_args.append("--train-model1")
    if TRAIN_MODEL_2_HUMAN:
        cmd_args.append("--train-model2")

if RUN_MODE == "CONTINUOUS_DAEMON_4H":
    cmd_args.append(f"--loop {LOOP_INTERVAL}")
    if ANTI_IDLE_HEARTBEAT:
        cmd_args.append("--keep-alive")

# Gọi file thực thi bằng đường dẫn tuyệt đối đảm bảo 100% không bị lỗi directory
script_path = Path("/content/DeepFake-Detection-Agent/colab_sync_to_gdrive.py")
if not script_path.exists():
    # Thử tìm kiếm trong thư mục hiện tại
    script_path = Path("colab_sync_to_gdrive.py").resolve()

cmd_str = f"python {script_path} " + " ".join(cmd_args)
print(f"🚀 Đang thực thi: {cmd_str}\n")
!{cmd_str}

--- 
## Bước 5: Báo Cáo Huấn Luyện Toàn Diện (Report Data Đầy Đủ)

In [ ]:
from pathlib import Path
import IPython.display as display

report_path = Path("report_colab/training_report_dual_stream.md")
gdrive_report = Path("/content/drive/MyDrive/Thread_Data/reports/training_report_dual_stream.md")

target_report = gdrive_report if gdrive_report.exists() else report_path

if target_report.exists():
    print(f"📋 Hiển thị Báo Cáo Huấn Luyện Toàn Diện từ: {target_report}\n")
    with open(target_report, "r", encoding="utf-8") as f:
        report_content = f.read()
    display.display(display.Markdown(report_content))
else:
    print("⚠️ Chưa tìm thấy file báo cáo training_report_dual_stream.md. Vui lòng kiểm tra lại log thực thi.")

# Kiểm tra các tệp checkpoint trọng số đã lưu trên Google Drive
print("\n" + "="*65)
print("💾 DANH MỤC TRỌNG SỐ ĐÃ LƯU TRÊN GOOGLE DRIVE /content/drive/MyDrive/Thread_Data/checkpoints/")
print("="*65)
ckpt_dir = Path("/content/drive/MyDrive/Thread_Data/checkpoints")
if ckpt_dir.exists():
    for p in ckpt_dir.iterdir():
        size_mb = round(p.stat().st_size / (1024 * 1024), 2)
        print(f"  - [{p.name}] Kích thước: {size_mb} MB")
else:
    print("  (Đang chạy local - kiểm tra thư mục report_colab/gdrive_storage/checkpoints/)")
print("="*65)